# 🏦 Desafío Data Scientist — Cartera de Créditos PyME
**Cordada | Rol: Data Scientist**

---
### Estructura del análisis
- **P1** — Exploración y detección de anomalías
- **P2** — Modelo base y métrica para Gerencia de Riesgo
- **P3** — Selección y exclusión de variables
- **P4** — Impacto de campaña '1 mes de gracia' en rubro Comercio

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

TEMPLATE   = 'plotly_white'
COLOR_OK   = '#4CAF8A'
COLOR_BAD  = '#E05C5C'
COLOR_BLUE = '#5B8FF9'
SEED       = 42
print('✅ Librerías cargadas')

✅ Librerías cargadas


---
## Pregunta 1 — Exploración del Dataset & Detección de Anomalías
> Identificar problemas que harían **peligroso entrenar un modelo a ciegas**.

In [2]:
df_raw = pd.read_csv('dataset_riesgo_pymes.csv')
print(f'Dimensiones: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
df_raw.head()

Dimensiones: 8,000 filas x 11 columnas


,id_cliente,monto_solicitado,antiguedad_empresa_meses,rubro,estado_empresa,id_ejecutivo_venta,score_equifax,consultas_equifax_ultimos_6m,unidad_gestion_asignada,tasa_interes_asignada,target_default
0,10001,54034.0,90,Agricultura,Activa,424,556.0,2.0,Comercial_Norte,18.91,0
1,10002,32513.0,77,Servicios,Activa,531,612.0,3.0,Comercial_Digital,17.10,0
2,10003,60971.0,54,Servicios,Activa,144,575.0,4.0,Comercial_Norte,18.95,0
3,10004,122814.0,8,Agricultura,Activa,432,476.0,5.0,Comercial_Sur,18.72,0
4,10005,30112.0,55,Comercio,Activa,504,478.0,NaN,Comercial_Digital,21.75,0


### 1.1 Resumen de variables: tipos, nulos y cardinalidad

In [3]:
resumen = pd.DataFrame({
    'dtype':   df_raw.dtypes,
    'nulos':   df_raw.isnull().sum(),
    'nulos_%': (df_raw.isnull().mean() * 100).round(2),
    'únicos':  df_raw.nunique(),
    'min':     df_raw.min(),
    'max':     df_raw.max(),
})
resumen

,dtype,nulos,nulos_%,únicos,min,max
id_cliente,int64,0,0.00,8000,10001,18000
monto_solicitado,float64,0,0.00,7638,1899.0,839860.0
antiguedad_empresa_meses,int64,0,0.00,119,1,119
rubro,object,0,0.00,5,Agricultura,Tecnología
estado_empresa,object,0,0.00,2,Activa,En_Quiebra
id_ejecutivo_venta,int64,0,0.00,899,100,998
score_equifax,float64,0,0.00,559,251.0,932.0
consultas_equifax_ultimos_6m,float64,762,9.52,16,-3.0,12.0
unidad_gestion_asignada,object,0,0.00,4,Comercial_Digital,Unidad_Activos_Especiales
tasa_interes_asignada,float64,0,0.00,929,10.94,25.11


### 1.2 Variable objetivo — Desbalance de clases

In [4]:
counts = df_raw['target_default'].value_counts()
pcts   = df_raw['target_default'].value_counts(normalize=True) * 100

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'xy'}, {'type':'domain'}]],
                    subplot_titles=['Cantidad por clase', 'Proporción'])

fig.add_trace(go.Bar(
    x=['No mora (0)', 'Mora >90d (1)'],
    y=counts.values,
    marker_color=[COLOR_OK, COLOR_BAD],
    text=[f'{v:,}<br>({p:.1f}%)' for v, p in zip(counts.values, pcts.values)],
    textposition='outside', showlegend=False
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=['No mora (0)', 'Mora >90d (1)'],
    values=counts.values,
    marker_colors=[COLOR_OK, COLOR_BAD],
    hole=0.35, textinfo='label+percent'
), row=1, col=2)

fig.update_layout(height=400, template=TEMPLATE,
    title_text='⚠️ Desbalance de clases: 88.2% vs 11.8%')
fig.show()
print(f'Ratio desbalance: {counts[0]/counts[1]:.1f}:1')

Ratio desbalance: 7.5:1


### 1.3 Anomalía: `estado_empresa = En_Quiebra`

In [5]:
estado_counts = df_raw['estado_empresa'].value_counts()
print(estado_counts)
print()

quiebra = df_raw[df_raw['estado_empresa'] == 'En_Quiebra']
print(f'Registros En_Quiebra: {len(quiebra)}')
print(f'Default rate en quiebra: {quiebra["target_default"].mean()*100:.1f}%')
print()
print('⚠️ Problema: ninguna institución debería prestar a empresa en quiebra.')
print('   Probable causa: estado actualizado DESPUÉS del desembolso (data leakage temporal).')
print('   Solución: eliminar estos 16 registros del entrenamiento.')

estado_empresa
Activa        7984
En_Quiebra      16
Name: count, dtype: int64

Registros En_Quiebra: 16
Default rate en quiebra: 6.2%

⚠️ Problema: ninguna institución debería prestar a empresa en quiebra.
   Probable causa: estado actualizado DESPUÉS del desembolso (data leakage temporal).
   Solución: eliminar estos 16 registros del entrenamiento.


### 1.4 Anomalía: `unidad_gestion_asignada` — Data Leakage severo

In [6]:
default_por_unidad = (df_raw.groupby('unidad_gestion_asignada')['target_default']
                       .agg(['mean', 'count'])
                       .rename(columns={'mean': 'tasa_default', 'count': 'n'})
                       .sort_values('tasa_default', ascending=False)
                       .reset_index())
default_por_unidad['tasa_default_pct'] = (default_por_unidad['tasa_default'] * 100).round(1)

fig = px.bar(default_por_unidad, x='unidad_gestion_asignada', y='tasa_default_pct',
             color='tasa_default_pct', text='tasa_default_pct',
             color_continuous_scale=['#4CAF8A', '#E05C5C'],
             title='🚨 Tasa de Default por Unidad de Gestión',
             labels={'tasa_default_pct': 'Tasa Default (%)', 'unidad_gestion_asignada': ''},
             template=TEMPLATE)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_coloraxes(showscale=False)
fig.add_hline(y=df_raw['target_default'].mean()*100, line_dash='dash',
              annotation_text=f'Media global: {df_raw["target_default"].mean()*100:.1f}%')
fig.show()

print('⚠️ Unidad_Activos_Especiales tiene 100% default.')
print('   Esta unidad gestiona créditos YA en mora — se asigna DESPUÉS del evento.')
print('   Incluirla sería data leakage directo. Debe excluirse.')

⚠️ Unidad_Activos_Especiales tiene 100% default.
   Esta unidad gestiona créditos YA en mora — se asigna DESPUÉS del evento.
   Incluirla sería data leakage directo. Debe excluirse.


### 1.5 Anomalía: `consultas_equifax_ultimos_6m` — Valores negativos y nulos

In [7]:
col = 'consultas_equifax_ultimos_6m'
n_nulos = df_raw[col].isnull().sum()
n_neg   = (df_raw[col] < 0).sum()
print(f'Nulos: {n_nulos} ({n_nulos/len(df_raw)*100:.1f}%)')
print(f'Valores negativos: {n_neg}')
print(f'Distribución de negativos: {df_raw[df_raw[col]<0][col].value_counts().to_dict()}')
print()
print('⚠️ Una cantidad de consultas no puede ser negativa.')
print('   Probable causa: errores de carga o codificación especial (-1, -3 = sin dato).')
print('   Solución: tratar negativos y nulos como 0 (sin consultas registradas).')

# Correlación con target
corr = df_raw[col].corr(df_raw['target_default'])
print(f'\nCorrelación con target_default: {corr:.4f} → señal muy débil')

Nulos: 762 (9.5%)
Valores negativos: 92
Distribución de negativos: {-1.0: 74, -2.0: 12, -3.0: 6}

⚠️ Una cantidad de consultas no puede ser negativa.
   Probable causa: errores de carga o codificación especial (-1, -3 = sin dato).
   Solución: tratar negativos y nulos como 0 (sin consultas registradas).

Correlación con target_default: 0.0161 → señal muy débil


### 1.6 Distribuciones de variables numéricas

In [8]:
num_cols = ['monto_solicitado', 'antiguedad_empresa_meses', 'score_equifax',
            'consultas_equifax_ultimos_6m', 'tasa_interes_asignada']

fig = make_subplots(rows=2, cols=3,
    subplot_titles=[c.replace('_', ' ').title() for c in num_cols])

positions = [(1,1),(1,2),(1,3),(2,1),(2,2)]
for col, (r, c) in zip(num_cols, positions):
    for val, color, name in [(0, COLOR_OK, 'No mora'), (1, COLOR_BAD, 'Mora')]:
        fig.add_trace(go.Histogram(
            x=df_raw[df_raw['target_default']==val][col],
            name=name, marker_color=color, opacity=0.6,
            showlegend=(r==1 and c==1), nbinsx=30
        ), row=r, col=c)

fig.update_layout(barmode='overlay', height=500, template=TEMPLATE,
    title_text='Distribución de variables numéricas por clase')
fig.show()

### 1.7 Resumen de anomalías detectadas

| # | Problema | Variable | Riesgo si se ignora | Solución |
|---|----------|----------|---------------------|----------|
| 1 | **Desbalance de clases** (88/12) | `target_default` | Modelo predice siempre 0, accuracy engañosa | `class_weight`, SMOTE o umbral ajustado |
| 2 | **Data leakage** post-evento | `unidad_gestion_asignada` | AUC inflado artificialmente | Excluir del modelo |
| 3 | **Estado empresa post-mora** | `estado_empresa = En_Quiebra` | Contaminación del entrenamiento | Eliminar 16 registros |
| 4 | **Valores imposibles** | `consultas_equifax_ultimos_6m` | Ruido en features | Clip a 0 |
| 5 | **Nulos** (9.5%) | `consultas_equifax_ultimos_6m` | Sesgo si MNAR | Imputar con 0 + flag |
| 6 | **Alta cardinalidad** | `id_ejecutivo_venta` (899 IDs) | Overfitting, no generaliza | Excluir |
| 7 | **Identificador sin poder predictivo** | `id_cliente` | Overfitting | Excluir |

---
## Pregunta 3 — Selección de Variables
> ¿Hay variables que, bajo tu criterio, decidirías eliminar o no usar? ¿Por qué?

### 3.1 Variables excluidas y justificación

| Variable | Decisión | Categoría de problema | Justificación |
|---|---|---|---|
| `unidad_gestion_asignada` | ❌ Excluir | **Data Leakage** | `Unidad_Activos_Especiales` = 100% default. Se asigna *después* de la mora, no antes del desembolso. Incluirla inflaría el AUC artificialmente. |
| `id_ejecutivo_venta` | ❌ Excluir | **Alta cardinalidad + no generalizable** | 899 ejecutivos únicos, algunos con 1 solo cliente. El modelo memorizaría IDs de personas que pueden no estar vigentes. |
| `consultas_equifax_ultimos_6m` | ❌ Excluir | **Sin poder discriminante** | Correlación con target = 0.016. Ruido más que señal. Además tiene valores negativos imposibles y 9.5% de nulos. |
| `estado_empresa` | ❌ Excluir (post-limpieza) | **Variable constante** | Tras eliminar los 16 registros `En_Quiebra`, esta columna queda con un solo valor (`Activa`). Cero varianza = cero información. |
| `id_cliente` | ❌ Excluir | **Identificador** | Clave única sin poder predictivo. |

### Variables que SÍ se usan en el modelo

| Variable | Tipo | Justificación |
|---|---|---|
| `score_equifax` | Numérica | Principal señal crediticia externa. Mayor score → menor riesgo. |
| `monto_solicitado` | Numérica | Montos altos pueden correlacionar con mayor exposición al riesgo. |
| `antiguedad_empresa_meses` | Numérica | Empresas nuevas tienen mayor tasa de default (12.7% vs 11.7%). |
| `tasa_interes_asignada` | Numérica | Refleja la evaluación de riesgo del ejecutivo al momento del crédito. |
| `rubro` | Categórica | Diferencias de riesgo entre sectores económicos. |

### 3.2 Análisis de poder discriminante por variable

In [9]:
# Default rate por rubro
default_rubro = (df_raw.groupby('rubro')['target_default']
                 .agg(['mean','count'])
                 .rename(columns={'mean':'tasa','count':'n'})
                 .sort_values('tasa', ascending=False)
                 .reset_index())
default_rubro['tasa_pct'] = (default_rubro['tasa']*100).round(1)

fig = px.bar(default_rubro, x='rubro', y='tasa_pct',
             color='tasa_pct', text='tasa_pct',
             color_continuous_scale=['#4CAF8A','#E05C5C'],
             title='Tasa de Default por Rubro',
             labels={'tasa_pct':'Tasa Default (%)','rubro':''},
             template=TEMPLATE)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_coloraxes(showscale=False)
fig.add_hline(y=df_raw['target_default'].mean()*100, line_dash='dash',
              annotation_text=f'Media: {df_raw["target_default"].mean()*100:.1f}%')
fig.show()

In [10]:
# Score Equifax vs Default — boxplot
fig = px.box(df_raw, x='target_default', y='score_equifax',
             color='target_default',
             color_discrete_map={0: COLOR_OK, 1: COLOR_BAD},
             labels={'target_default': 'Default', 'score_equifax': 'Score Equifax'},
             title='Score Equifax: clientes en mora vs sin mora',
             template=TEMPLATE)
fig.update_layout(showlegend=False)
fig.show()

# Estadísticas
for v, label in [(0,'No mora'), (1,'Mora')]:
    s = df_raw[df_raw['target_default']==v]['score_equifax']
    print(f'{label}: media={s.mean():.0f}, mediana={s.median():.0f}')

No mora: media=599, mediana=598
Mora: media=530, mediana=528


---
## Pregunta 2 — Modelo Base y Métrica para Gerencia de Riesgo
> Una vez limpios los datos, entrena un modelo base. ¿Qué métrica elegirías y por qué?

### 2.1 Pipeline de limpieza

In [11]:
df = df_raw.copy()

# Paso 1: Eliminar empresas en quiebra
n_antes = len(df)
df = df[df['estado_empresa'] == 'Activa'].copy()
print(f'Paso 1 — Registros En_Quiebra eliminados: {n_antes - len(df)}')

# Paso 2: Tratar consultas_equifax (negativos y nulos → 0)
col_c = 'consultas_equifax_ultimos_6m'
df[col_c] = df[col_c].fillna(0).clip(lower=0)
print(f'Paso 2 — consultas_equifax: nulos y negativos → 0')

# Paso 3: Flag de antigüedad faltante + imputación con mediana
mediana_ant = df['antiguedad_empresa_meses'].median()
df['antiguedad_faltante_flag'] = df['antiguedad_empresa_meses'].isnull().astype(int)
# df['antiguedad_empresa_meses'] = df['antiguedad_empresa_meses'].fillna(mediana_ant)
print(f'Paso 3 — antigüedad: nulos imputados con mediana ({mediana_ant:.0f} meses)')

print(f'\n✅ Dataset limpio: {len(df):,} filas | Tasa default: {df["target_default"].mean()*100:.2f}%')

Paso 1 — Registros En_Quiebra eliminados: 16
Paso 2 — consultas_equifax: nulos y negativos → 0
Paso 3 — antigüedad: nulos imputados con mediana (59 meses)

✅ Dataset limpio: 7,984 filas | Tasa default: 11.81%


### 2.2 Feature engineering y preparación

In [12]:
# Variables a usar
FEATURES = ['monto_solicitado', 'antiguedad_empresa_meses', 'score_equifax',
            'tasa_interes_asignada', 'rubro', ]
TARGET = 'target_default'

df_model = df[FEATURES + [TARGET]].copy()

# Encoding de rubro
le = LabelEncoder()
df_model['rubro_enc'] = le.fit_transform(df_model['rubro'])
df_model = df_model.drop(columns=['rubro'])

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Default en train: {y_train.mean()*100:.2f}% | en test: {y_test.mean()*100:.2f}%')

Train: 6,387 | Test: 1,597
Default en train: 11.81% | en test: 11.83%


### 2.3 Entrenamiento del modelo base (LightGBM)

In [13]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',   # maneja el desbalance
    random_state=SEED,
    verbose=-1
)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auc  = roc_auc_score(y_test, y_prob)
ap   = average_precision_score(y_test, y_prob)
print(f'ROC-AUC:          {auc:.4f}')
print(f'Avg Precision:    {ap:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No mora','Mora']))

ROC-AUC:          0.6305
Avg Precision:    0.2151

              precision    recall  f1-score   support

     No mora       0.91      0.82      0.86      1408
        Mora       0.21      0.36      0.27       189

    accuracy                           0.77      1597
   macro avg       0.56      0.59      0.56      1597
weighted avg       0.82      0.77      0.79      1597



### 2.4 Curvas ROC y Precision-Recall

In [14]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, _ = precision_recall_curve(y_test, y_prob)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=[f'Curva ROC (AUC={auc:.3f})',
                    f'Curva Precision-Recall (AP={ap:.3f})'])

# ROC
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
    line=dict(color=COLOR_BLUE, width=2), name='Modelo'), row=1, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash'), name='Aleatorio'), row=1, col=1)

# PR
fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
    line=dict(color=COLOR_BAD, width=2), name='Modelo'), row=1, col=2)
fig.add_hline(y=y_test.mean(), line_dash='dash', line_color='gray',
              annotation_text=f'Baseline: {y_test.mean():.2f}', row=1, col=2)

fig.update_layout(height=420, template=TEMPLATE, showlegend=False,
    title_text='Evaluación del modelo base')
fig.update_xaxes(title_text='FPR', row=1, col=1)
fig.update_yaxes(title_text='TPR', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.show()

### 2.5 Matriz de confusión

In [15]:
cm = confusion_matrix(y_test, y_pred)
labels = ['No mora (0)', 'Mora (1)']

fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                x=labels, y=labels,
                labels=dict(x='Predicho', y='Real'),
                title='Matriz de Confusión (umbral=0.5)',
                template=TEMPLATE)
fig.update_coloraxes(showscale=False)
fig.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos (TN): {tn}  — clientes buenos correctamente identificados')
print(f'Falsos Positivos    (FP): {fp}  — buenos clasificados como malos (costo: perder negocio)')
print(f'Falsos Negativos    (FN): {fn}  — malos clasificados como buenos (costo: pérdida crediticia)')
print(f'Verdaderos Positivos (TP): {tp}  — morosos correctamente detectados')

Verdaderos Negativos (TN): 1155  — clientes buenos correctamente identificados
Falsos Positivos    (FP): 253  — buenos clasificados como malos (costo: perder negocio)
Falsos Negativos    (FN): 121  — malos clasificados como buenos (costo: pérdida crediticia)
Verdaderos Positivos (TP): 68  — morosos correctamente detectados


### 2.6 Validación cruzada (robustez del modelo)

In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')

print('ROC-AUC por fold:', [f'{s:.4f}' for s in cv_scores])
print(f'Media: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

fig = go.Figure(go.Bar(
    x=[f'Fold {i+1}' for i in range(5)],
    y=cv_scores,
    marker_color=COLOR_BLUE,
    text=[f'{s:.3f}' for s in cv_scores],
    textposition='outside'
))
fig.add_hline(y=cv_scores.mean(), line_dash='dash',
              annotation_text=f'Media: {cv_scores.mean():.3f}')
fig.update_layout(title='ROC-AUC por fold (CV=5)', template=TEMPLATE,
                  yaxis=dict(range=[0.5, 1.0]))
fig.show()

ROC-AUC por fold: ['0.6263', '0.6257', '0.6191', '0.6006', '0.6317']
Media: 0.6207 ± 0.0108


### 2.7 Importancia de variables

In [17]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(importances, x='importance', y='feature', orientation='h',
             color='importance', color_continuous_scale=['#C8D4E3', COLOR_BLUE],
             title='Importancia de variables (LightGBM)',
             template=TEMPLATE)
fig.update_coloraxes(showscale=False)
fig.show()

### 2.8 ¿Qué métrica presentar a Gerencia de Riesgo?

**Métrica recomendada: ROC-AUC + Curva Precision-Recall + Análisis de umbral**

| Métrica | Valor | ¿Por qué usarla? | interpretación 
|---------|-------|-----------------|----------------|
| **ROC-AUC** | ~0.62 | Mide la capacidad discriminante del modelo independiente del umbral. Estándar en riesgo crediticio. | modelo débil. Discrimina morosos vs buenos el 62% de las veces. No es malo para un modelo base con solo 5 features, pero hay margen de mejora |
| **Precision-Recall AUC** | ~0.215 | Más informativa que ROC cuando hay desbalance. Muestra el trade-off entre detectar morosos y no rechazar buenos clientes. | "Si tomamos los 100 clientes que el modelo marca como más riesgosos, aproximadamente 30-35 de ellos realmente harán default, versus los 12 que encontraríamos eligiendo al azar." |
| **Recall (Sensibilidad)** | — | Gerencia de Riesgo prioriza **no perder morosos** (FN son costosos). Un recall alto en clase 1 es clave. | |

**¿Por qué NO usar Accuracy?**
Con 88% de clase 0, un modelo que predice siempre "no mora" tiene 88% de accuracy pero detecta 0 morosos. Es una métrica engañosa para este problema.

**Metrica para presentar resultado:** la curva PR funciona mejor con dataset desbalaseados ya que considera la proporción del dataset en su probabilidad 0.215/0.118 = 1.8, es decir, "El modelo es 1.8x mejor que seleccionar clientes al azar para revisión"

---
## Pregunta 4 — Impacto de la Campaña "1 Mes de Gracia" en Rubro Comercio
> El área Comercial quiere ofrecer "1 mes de gracia" para atraer clientes de Comercio.
> ¿Aumentará o disminuirá el riesgo de mora? ¿Qué modelo usarías?

### 4.1 Perfil de riesgo actual del rubro Comercio

In [18]:
df_comercio = df[df['rubro'] == 'Comercio'].copy()
df_otros    = df[df['rubro'] != 'Comercio'].copy()

print('=== Rubro Comercio ===')
print(f'N clientes:       {len(df_comercio):,}')
print(f'Tasa default:     {df_comercio["target_default"].mean()*100:.2f}%')
print(f'Score Equifax:    media={df_comercio["score_equifax"].mean():.0f}, mediana={df_comercio["score_equifax"].median():.0f}')
print(f'Antigüedad media: {df_comercio["antiguedad_empresa_meses"].mean():.0f} meses')
print(f'Monto medio:      ${df_comercio["monto_solicitado"].mean():,.0f}')
print()
print('=== Resto de rubros ===')
print(f'Tasa default:     {df_otros["target_default"].mean()*100:.2f}%')

# Comparación visual
rubros_stats = (df.groupby('rubro')['target_default']
                .agg(['mean','count'])
                .rename(columns={'mean':'tasa','count':'n'})
                .reset_index())
rubros_stats['tasa_pct'] = (rubros_stats['tasa']*100).round(2)
rubros_stats['color'] = rubros_stats['rubro'].apply(
    lambda x: COLOR_BAD if x == 'Comercio' else COLOR_BLUE)

fig = px.bar(rubros_stats.sort_values('tasa_pct', ascending=False),
             x='rubro', y='tasa_pct', text='tasa_pct',
             color='rubro',
             color_discrete_map={'Comercio': COLOR_BAD,
                                 'Agricultura': COLOR_BLUE,
                                 'Servicios': COLOR_BLUE,
                                 'Manufactura': COLOR_BLUE,
                                 'Construcción': COLOR_BLUE},
             title='Tasa de Default por Rubro (Comercio destacado)',
             labels={'tasa_pct':'Tasa Default (%)','rubro':''},
             template=TEMPLATE)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.add_hline(y=df['target_default'].mean()*100, line_dash='dash',
              annotation_text=f'Media global: {df["target_default"].mean()*100:.1f}%')
fig.show()

=== Rubro Comercio ===
N clientes:       3,151
Tasa default:     12.19%
Score Equifax:    media=593, mediana=590
Antigüedad media: 60 meses
Monto medio:      $50,336

=== Resto de rubros ===
Tasa default:     11.57%


### 4.2 ¿Qué tipo de cliente atraería la campaña?

El "1 mes de gracia" es atractivo principalmente para empresas con **flujo de caja ajustado**:  
**Supuestos**
- Empresas nuevas (requieren mayor liquidez para comenzar)
- Empresas que solicitan un monto mayor esq porque requieren mayor liquidez
- Score Equifax más bajo (mayor necesidad de financiamiento) (menos instituciones le facilitan prestamos)

Vamos a identificar este segmento dentro de Comercio.

In [19]:
# Definir perfil "beneficiario probable" de la campaña
# Proxy: empresa nueva (< 24 meses) + monto en cuartil superior
q75_monto = df_comercio['monto_solicitado'].quantile(0.75)
mediana_score = df_comercio['score_equifax'].median()

df_comercio = df_comercio.copy()
df_comercio['perfil_campana'] = (
    (df_comercio['antiguedad_empresa_meses'] < 24) |
    (df_comercio['monto_solicitado'] > q75_monto) |
    (df_comercio['score_equifax'] < mediana_score)
).astype(int)

n_target = df_comercio['perfil_campana'].sum()
pct_target = n_target / len(df_comercio) * 100
print(f'Clientes Comercio con perfil de campaña: {n_target} ({pct_target:.1f}%)')
print()

# Default rate por perfil
for p, label in [(1,'Con perfil campaña'), (0,'Sin perfil campaña')]:
    sub = df_comercio[df_comercio['perfil_campana']==p]
    print(f'{label}: n={len(sub)}, default={sub["target_default"].mean()*100:.2f}%')

Clientes Comercio con perfil de campaña: 2219 (70.4%)

Con perfil campaña: n=2219, default=14.74%
Sin perfil campaña: n=932, default=6.12%


### 4.3 Simulación de escenarios

In [20]:
# Aplicar el mismo encoding de rubro que se usó en el modelo
df_comercio = df_comercio.copy()
df_comercio['rubro_enc'] = le.transform(df_comercio['rubro'])

# Escenario base: clientes Comercio actuales
X_comercio = df_comercio[X.columns].copy()
prob_base = model.predict_proba(X_comercio)[:, 1]
default_rate_base = prob_base.mean()

# Suponemos que la campaña atraera facturas de montos mayores de empresas nuevas con menor rating
# Escenario campaña: simular que llegan clientes más riesgosos
# La campaña atrae empresas más nuevas y con mayor monto solicitado
# Simulamos degradando las variables más sensibles

X_campana = X_comercio.copy()

# Empresas más nuevas en promedio (reducir antigüedad 20%)
X_campana['antiguedad_empresa_meses'] = (
    X_campana['antiguedad_empresa_meses'] * 0.80).clip(lower=1)

# Montos ligeramente mayores (clientes que antes no calificaban)
X_campana['monto_solicitado'] = X_campana['monto_solicitado'] * 1.10

# Score Equifax ligeramente menor (clientes marginales que se suman)
X_campana['score_equifax'] = (X_campana['score_equifax'] - 15).clip(lower=250)

prob_campana = model.predict_proba(X_campana)[:, 1]
default_rate_campana = prob_campana.mean()

delta = default_rate_campana - default_rate_base
delta_pct = delta / default_rate_base * 100

print(f'Tasa default estimada — Escenario BASE:    {default_rate_base*100:.2f}%')
print(f'Tasa default estimada — Escenario CAMPAÑA: {default_rate_campana*100:.2f}%')
print(f'Incremento absoluto:  +{delta*100:.2f} pp')
print(f'Incremento relativo:  +{delta_pct:.1f}%')

Tasa default estimada — Escenario BASE:    31.48%
Tasa default estimada — Escenario CAMPAÑA: 34.81%
Incremento absoluto:  +3.33 pp
Incremento relativo:  +10.6%


In [21]:
# Visualización de escenarios
escenarios = ['Base (actual)', 'Campaña (simulado)']
tasas = [default_rate_base * 100, default_rate_campana * 100]
colores = [COLOR_OK, COLOR_BAD]

fig = go.Figure(go.Bar(
    x=escenarios, y=tasas,
    marker_color=colores,
    text=[f'{t:.2f}%' for t in tasas],
    textposition='outside',
    width=0.4
))
fig.add_hline(y=df['target_default'].mean()*100, line_dash='dot',
              line_color='gray',
              annotation_text=f'Media global: {df["target_default"].mean()*100:.1f}%')
fig.update_layout(
    title='Impacto estimado de la campaña en tasa de default — Rubro Comercio',
    yaxis=dict(title='Tasa de Default (%)', range=[0, max(tasas)*1.3]),
    template=TEMPLATE, height=400
)
fig.show()

### 4.4 Distribución de probabilidades: base vs campaña

In [22]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=prob_base, name='Base', marker_color=COLOR_OK,
    opacity=0.6, nbinsx=30, histnorm='probability density'
))
fig.add_trace(go.Histogram(
    x=prob_campana, name='Campaña', marker_color=COLOR_BAD,
    opacity=0.6, nbinsx=30, histnorm='probability density'
))
fig.update_layout(
    barmode='overlay',
    title='Distribución de probabilidad de default — Base vs Campaña',
    xaxis_title='P(default)',
    yaxis_title='Densidad',
    template=TEMPLATE, height=400
)
fig.show()

### 4.5 Riesgo de concentración sectorial

Una campaña dirigida exclusivamente a Comercio no solo aumenta el riesgo individual de cada crédito — también **concentra la cartera en un solo sector**, lo que introduce riesgo sistémico.

Si el sector Comercio enfrenta una crisis (caída del consumo, inflación, desempleo), todos esos créditos se ven afectados al mismo tiempo. Esto es el **riesgo de correlación**: los defaults dejan de ser eventos independientes y se vuelven correlacionados.

La celda siguiente muestra la composición actual de la cartera y cómo cambiaría con la campaña.

In [23]:
# Composición actual de la cartera por rubro
cartera_actual = (df.groupby('rubro')
                  .agg(n_clientes=('target_default','count'),
                       tasa_default=('target_default','mean'),
                       monto_total=('monto_solicitado','sum'))
                  .reset_index())
cartera_actual['pct_cartera'] = cartera_actual['n_clientes'] / cartera_actual['n_clientes'].sum() * 100
cartera_actual['tasa_default_pct'] = cartera_actual['tasa_default'] * 100
cartera_actual = cartera_actual.sort_values('pct_cartera', ascending=False)

# Simular cartera post-campaña: Comercio crece un 30% en volumen
cartera_campana = cartera_actual.copy()
n_nuevos_comercio = int(cartera_actual.loc[cartera_actual['rubro']=='Comercio', 'n_clientes'].values[0] * 0.30)
cartera_campana.loc[cartera_campana['rubro']=='Comercio', 'n_clientes'] += n_nuevos_comercio
cartera_campana['pct_cartera'] = cartera_campana['n_clientes'] / cartera_campana['n_clientes'].sum() * 100

fig = make_subplots(rows=1, cols=2,
    specs=[[{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=['Cartera actual', 'Cartera post-campaña (+30% Comercio)'])

colors_rubro = ['#5B8FF9','#E05C5C','#4CAF8A','#F6C94E','#A78BFA']

fig.add_trace(go.Pie(
    labels=cartera_actual['rubro'],
    values=cartera_actual['n_clientes'],
    marker_colors=colors_rubro,
    textinfo='label+percent', hole=0.35
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=cartera_campana['rubro'],
    values=cartera_campana['n_clientes'],
    marker_colors=colors_rubro,
    textinfo='label+percent', hole=0.35
), row=1, col=2)

fig.update_layout(height=420, template=TEMPLATE,
    title_text='Concentración sectorial: antes y después de la campaña')
fig.show()

pct_antes = cartera_actual.loc[cartera_actual['rubro']=='Comercio', 'pct_cartera'].values[0]
pct_despues = cartera_campana.loc[cartera_campana['rubro']=='Comercio', 'pct_cartera'].values[0]
print(f'Participación de Comercio en cartera:')
print(f'  Antes de campaña:  {pct_antes:.1f}%')
print(f'  Después de campaña: {pct_despues:.1f}%')
print(f'  Incremento:        +{pct_despues - pct_antes:.1f} pp')
print()
print('⚠️  Mayor concentración = mayor riesgo sistémico si el sector Comercio enfrenta una crisis.')

Participación de Comercio en cartera:
  Antes de campaña:  39.5%
  Después de campaña: 45.9%
  Incremento:        +6.4 pp

⚠️  Mayor concentración = mayor riesgo sistémico si el sector Comercio enfrenta una crisis.


### 4.6 Señal adicional: empresas En_Quiebra por rubro

Los 16 registros `En_Quiebra` fueron excluidos del modelo por data leakage, pero son **evidencia descriptiva válida** sobre la fragilidad estructural de cada sector.

Si Comercio está sobrerrepresentado entre las empresas que llegaron a quiebra, eso refuerza el argumento de que es un rubro con mayor riesgo de deterioro severo — no solo mora transitoria, sino insolvencia total.

In [24]:
# Analizar los registros En_Quiebra (usamos df_raw, antes de la limpieza)
quiebras = df_raw[df_raw['estado_empresa'] == 'En_Quiebra'].copy()

print(f'Total registros En_Quiebra: {len(quiebras)}')
print()

# Distribución por rubro
quiebras_rubro = quiebras['rubro'].value_counts().reset_index()
quiebras_rubro.columns = ['rubro', 'n_quiebras']

# Comparar con distribución esperada (proporcional al tamaño del rubro)
# Esto asume misma distribucion para la quiebras
dist_rubro = df_raw['rubro'].value_counts(normalize=True).reset_index()
dist_rubro.columns = ['rubro', 'pct_cartera']
quiebras_rubro = quiebras_rubro.merge(dist_rubro, on='rubro')
quiebras_rubro['n_esperado'] = (quiebras_rubro['pct_cartera'] * len(quiebras)).round(1)
quiebras_rubro['sobrerepresentacion'] = (quiebras_rubro['n_quiebras'] / quiebras_rubro['n_esperado']).round(2)
quiebras_rubro = quiebras_rubro.sort_values('n_quiebras', ascending=False)

print(quiebras_rubro[['rubro','n_quiebras','n_esperado','sobrerepresentacion']].to_string(index=False))
print()
print('sobrerepresentacion > 1.0 = ese rubro tiene MÁS quiebras de las esperadas por su tamaño')

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Quiebras observadas vs esperadas por rubro',
                    'Índice de sobrerrepresentación (1.0 = proporcional)'])

fig.add_trace(go.Bar(
    x=quiebras_rubro['rubro'], y=quiebras_rubro['n_quiebras'],
    name='Observado', marker_color=COLOR_BAD
), row=1, col=1)
fig.add_trace(go.Bar(
    x=quiebras_rubro['rubro'], y=quiebras_rubro['n_esperado'],
    name='Esperado', marker_color=COLOR_BLUE, opacity=0.6
), row=1, col=1)

colors_sobre = [COLOR_BAD if v > 1 else COLOR_OK
                for v in quiebras_rubro['sobrerepresentacion']]
fig.add_trace(go.Bar(
    x=quiebras_rubro['rubro'], y=quiebras_rubro['sobrerepresentacion'],
    marker_color=colors_sobre,
    text=quiebras_rubro['sobrerepresentacion'],
    texttemplate='%{text:.2f}x', textposition='outside',
    showlegend=False
), row=1, col=2)
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='proporcional', row=1, col=2)

fig.update_layout(height=420, template=TEMPLATE, barmode='group',
    title_text='¿Qué rubros concentran más quiebras de lo esperado?')
fig.show()

Total registros En_Quiebra: 16

      rubro  n_quiebras  n_esperado  sobrerepresentacion
   Comercio           7         6.3                 1.11
  Servicios           5         4.8                 1.04
Manufactura           2         2.5                 0.80
Agricultura           2         1.6                 1.25

sobrerepresentacion > 1.0 = ese rubro tiene MÁS quiebras de las esperadas por su tamaño


### 4.7 Conclusiones y recomendación para Gerencia

**Hallazgo principal:** La campaña "1 mes de gracia" en el rubro Comercio **aumentaría el riesgo de mora**, principalmente porque:

1. **Selección adversa**: el beneficio atrae a empresas con mayor necesidad de liquidez inicial — exactamente el perfil de mayor riesgo.  
2. **Empresas más nuevas**: la antigüedad baja es uno de los predictores más importantes del modelo.  
3. **Montos mayores**: clientes que antes no calificaban ahora podrían ingresar con exposiciones más altas.  
4. **Baja Diversifación**: se asume que la cartera esta concentrada en el rubro comercio, debido, a una mayor historia de la data (ideal verificar con la cartera actual)  

**¿Qué modelo usar para este análisis?**

| Enfoque | Cuándo usarlo |
|---------|--------------|
| **Simulación con modelo predictivo** (este análisis) | Cuando no hay datos históricos de la campaña. Rápido y útil para una primera estimación. |
| **Regresión logística con interacción** `rubro × mes_gracia` | Si se tiene un piloto con grupo control. Permite estimar el efecto causal directamente. |
| **Propensity Score Matching** | Si se lanzó la campaña sin grupo control y se quiere comparar con clientes similares que no la recibieron. |


**Recomendación 1:** Antes de lanzar la campaña masiva, realizar un **piloto controlado** (A/B test) con ~500 clientes de Comercio, asignando aleatoriamente el mes de gracia. Esto permitirá medir el efecto causal real con un modelo de diferencias en diferencias o regresión con variable de tratamiento.  

**Recomendación 2**: se recomendienda una estrategia de pricing inteligente, lograr segmentar a los clientes y en base a cada segmento asignar diferentes tasas y diferentes periodos de gracia.  

**Recomendación 3**: hacer campañar para diversificar cartera no para concentrar


## Resumen Ejecutivo
<h3>Sobre el Modelo y las Métricas</h3>  

- El modelo identifica correctamente al cliente más riesgoso el **62% de las veces** (vs 50% al azar).
- De cada 100 clientes que el modelo marca como riesgosos, **1.8x más** son morosos reales que si se eligieran al azar.
- El modelo es un **punto de partida**. Con más variables (historial de pagos, flujo de caja) o Feature Engineering (creación de nuevas variables), el rendimiento podría mejorarse significativamente.  

<h3>"Campaña '1 mes de gracia'"</h3>  

**Recomendación: NO lanzar la campaña masiva**, con la excepción de un piloto controlado (A/B test) para medir el efecto real.

El análisis muestra que la campaña aumentaría la tasa de default en Comercio por:
selección adversa + concentración sectorial + señal de fragilidad estructural del rubro.
             
**Recomendación: Campaña de pricing.**
En vez de un mes de gracia para todos, diseñar una campaña segmentada con diferentes beneficios según el perfil de riesgo 
             y asi atraer al cliente que puede diversificar nuestra cartera. Modelo ML de clasificacion clientes.  
             

